# LTIC: H22-clean training on Kaggle (Phase 3)

Trains the pre-registered H22-clean runs (`gates/h22_confirmatory.json`) on the features from `h22_extract`:
baselines `ce, la, crt, tau_norm, ltic2, ltic3`, then C2 `tau_norm+c2f` and its 3 shuffled-taxonomy (A9) controls.
One encoder per GPU, so **T4 x2 trains two encoders at once**.

* `MODE = "timing"`: 1 short trial per method on V-tune only, nothing kept. Checks everything works and measures speed.
* `MODE = "full"`: the real runs. **Refuses to start until the rules file says AGREED.**

Right-hand panel before running:
1. **Accelerator:** `GPU T4 x2`
2. **Internet:** On (to get the code)
3. **Input:** *Add Input → Your Work →* `h22_extract` (or the `h22clean-feats` Dataset). No competition data needed.
4. For a later session: also add **this notebook's previous version** as input (finished methods are skipped).

Then **Save Version → Save & Run All (Commit)**. It keeps running after you close the browser and stops by itself
before Kaggle's 12-hour limit.

In [ ]:
# ---- Settings: the only cell you need to edit ----
GITHUB_USER = "dgpro06"
BRANCH      = "Dgpro"

MODE        = "timing"                 # "timing" first; "full" only after the rules are AGREED and committed
ENCODERS    = "dinov2_b14,bioclip2"    # primary encoders, one per GPU; "clip_b32" alone uses both GPUs
DEADLINE_MIN = 600                     # stop starting new methods after 10 h (Kaggle kills at 12 h, keeps nothing);
                                       # keep it below your remaining GPU quota too (Kaggle stops at 0 and keeps nothing)
IJCNN       = 0                        # 1: also run the IJCNN 2025 configuration on clip_b32 (corrected H22-clean number)

## Step 1: check the GPU and the inputs

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!ls /kaggle/input

## Step 2: get the code from GitHub

In [ ]:
import os
CODE = "/kaggle/working/LTIC-Herb"
if os.path.isdir(CODE):
    !git -C {CODE} pull
else:
    !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/LTIC-Herb.git {CODE}
!git -C {CODE} log --oneline -1

## Step 3: train (`kaggle/h22_train.sh`). Progress is printed every 5 minutes.

In [ ]:
os.environ.update(MODE=MODE, ENCODERS=ENCODERS, DEADLINE_MIN=str(DEADLINE_MIN), IJCNN=str(IJCNN))
!bash {CODE}/kaggle/h22_train.sh
if get_ipython().user_ns.get("_exit_code", 0) != 0:
    raise RuntimeError("The run stopped with an error - scroll up to the first error message.")
!rm -rf {CODE}   # keep the Output small: results/ and logs/ only